In [16]:
# ====================================================================
# Smart Farming Pipeline - OpenCV Preprocessing & Context Notebook (v3)
# Process ALL images in testing directory
# ====================================================================

import copy
import os
from pathlib import Path
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import yaml
import pprint
import pandas as pd

# ====================================================================
# 1. Configuration Layer
# ====================================================================
config_data = """
models:
  crop_identifier: "models/crop_identifier_v2.pth"
  disease_models:
    tomato: "models/tomato_disease_v1.pth"
    potato: "models/potato_disease_v1.pth"
    cotton: "models/cotton_disease_v1.pth"
thresholds:
  crop_confidence: 0.75
  disease_confidence: 0.70
  blur_var_threshold: 100.0
  min_brightness: 40.0
  max_brightness: 240.0
storage:
  upload_dir: "uploads/"
  processed_dir: "processed/"
"""

with open("config.yaml", "w") as f:
    f.write(config_data)

with open("config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)

# ====================================================================
# 2. Shared Context Object
# ====================================================================
def create_context(image_path, user_id="0001", location="Bhavnagar"):
    return {
        "request_id": "RE0001",
        "user": {
            "user_id": user_id,
            "location": location,
            "language": "Gujarati",
        },
        "image": {
            "raw_path": str(image_path),
            "processed_path": None,
            "quality_score": None,
            "blur_score": None,
            "brightness_score": None,
            "leaf_detected": False,
        },
        "crop": {"label": None, "confidence": None},
        "disease": {"label": None, "confidence": None, "model_used": None},
        "severity": {"percent": None, "affected_area": None},
        "weather": {},
        "recommendation": {},
        "status": {
            "preprocessing": "pending",
            "crop_identification": "pending",
            "decision_routing": "pending",
            "disease_classification": "pending",
            "severity": "pending",
            "recommendation": "pending",
        },
    }

# ====================================================================
# 3. OpenCV Preprocessing & Validation Service
# ====================================================================
class OpenCVPreprocessorService:
    def __init__(self, config=CONFIG):
        self.config = config

    def process(self, context):
        raw_path = context["image"]["raw_path"]
        input_im = cv.imread(raw_path, cv.IMREAD_COLOR)

        if input_im is None:
            print(f"[Error] Failed to read image: {raw_path}")
            context["status"]["preprocessing"] = "failed_read"
            return context

        # A. Blur Detection
        gray = cv.cvtColor(input_im, cv.COLOR_BGR2GRAY)
        blur_score = cv.Laplacian(gray, cv.CV_64F).var()
        context["image"]["blur_score"] = float(blur_score)
        blur_threshold = self.config["thresholds"]["blur_var_threshold"]

        if blur_score < blur_threshold:
            print(f"[Validation Warning] Image too blurry (Score: {blur_score:.2f} < {blur_threshold})")
            context["image"]["leaf_detected"] = False
            context["status"]["preprocessing"] = "failed_blur"
            return context

        # B. Brightness Check
        hsv = cv.cvtColor(input_im, cv.COLOR_BGR2HSV)
        brightness_score = np.mean(hsv[:, :, 2])
        context["image"]["brightness_score"] = float(brightness_score)
        min_brightness = self.config["thresholds"]["min_brightness"]
        max_brightness = self.config["thresholds"]["max_brightness"]

        if not (min_brightness <= brightness_score <= max_brightness):
            print(f"[Validation Warning] Poor lighting (Brightness: {brightness_score:.2f})")
            context["image"]["leaf_detected"] = False
            context["status"]["preprocessing"] = "failed_lighting"
            return context

        # C. Leaf Detection
        low_H, high_H = 25, 85
        low_S, high_S = 30, 255
        low_V, high_V = 30, 255
        im_threshold = cv.inRange(hsv, (low_H, low_S, low_V), (high_H, high_S, high_V))

        kernel = np.ones((3, 3), np.uint8)
        opening = cv.morphologyEx(im_threshold, cv.MORPH_OPEN, kernel, iterations=2)
        sure_bg = cv.dilate(opening, kernel, iterations=3)

        dist_transform = cv.distanceTransform(opening, cv.DIST_L2, 5)
        max_distance = dist_transform.max()

        if max_distance == 0:
            print("[Validation Warning] No plant-like region found.")
            context["image"]["leaf_detected"] = False
            context["status"]["preprocessing"] = "failed_no_leaf"
            return context

        _, sure_fg = cv.threshold(dist_transform, 0.15 * max_distance, 255, 0)
        sure_fg = np.uint8(sure_fg)
        unknown = cv.subtract(sure_bg, sure_fg)

        # Watershed
        _, markers = cv.connectedComponents(sure_fg)
        markers = markers + 1
        markers[unknown == 255] = 0
        markers = cv.watershed(input_im, markers)

        # Contour and size verification
        valid_mask = np.zeros_like(im_threshold)
        leaf_found = False

        if markers.max() >= 2:
            for i in range(2, markers.max() + 1):
                component_mask = np.uint8(markers == i) * 255
                contours, _ = cv.findContours(component_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
                for cnt in contours:
                    area = cv.contourArea(cnt)
                    x, y, cw, ch = cv.boundingRect(cnt)
                    if area < 150 or cw < 30 or ch < 30:
                        continue
                    cv.drawContours(valid_mask, [cnt], -1, 255, thickness=cv.FILLED)
                    leaf_found = True

        if not leaf_found:
            print("[Validation Warning] No valid leaf contours detected.")
            context["image"]["leaf_detected"] = False
            context["status"]["preprocessing"] = "failed_no_leaf"
            return context

        # D. Background Removal & E. CLAHE Enhancement
        masked_img = input_im.copy()
        masked_img[valid_mask == 0] = [0, 0, 0]

        lab = cv.cvtColor(masked_img, cv.COLOR_BGR2LAB)
        l, a, b = cv.split(lab)
        clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)
        limg = cv.merge((cl, a, b))
        enhanced = cv.cvtColor(limg, cv.COLOR_LAB2BGR)

        # F. Resize to 224 x 224 & G. Save processed image
        final_processed = cv.resize(enhanced, (224, 224), interpolation=cv.INTER_AREA)
        processed_dir = self.config["storage"]["processed_dir"]
        os.makedirs(processed_dir, exist_ok=True)
        processed_filename = Path(raw_path).name
        processed_save_path = os.path.join(processed_dir, processed_filename)
        cv.imwrite(processed_save_path, final_processed)

        # H. Quality Score & I. Update Context
        quality_score = blur_score * 0.5 + brightness_score * 0.5
        context["image"]["processed_path"] = processed_save_path
        context["image"]["quality_score"] = float(quality_score)
        context["image"]["leaf_detected"] = True
        context["status"]["preprocessing"] = "completed"

        return context

# ====================================================================
# 4. Pipeline Orchestrator
# ====================================================================
def run_pipeline(image_path):
    context = create_context(image_path)
    preprocessor = OpenCVPreprocessorService(CONFIG)
    context = preprocessor.process(context)

    if context["status"]["preprocessing"] != "completed":
        print(f"[Pipeline Halted] Preprocessing failed with status: {context['status']['preprocessing']}")
        return context

    return context

# ====================================================================
# 5. Process ALL Images
# ====================================================================
def process_all_images(test_dir):
    test_dir = Path(test_dir)
    if not test_dir.exists():
        print(f"[Error] Testing directory not found:\n{test_dir}")
        return [], pd.DataFrame()

    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    image_paths = sorted([p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in image_extensions])

    if not image_paths:
        print("[Warning] No images found in testing directory.")
        return [], pd.DataFrame()

    print("=" * 80)
    print("SMART FARMING - BATCH IMAGE TEST")
    print("=" * 80)
    print(f"Testing directory : {test_dir}")
    print(f"Total images      : {len(image_paths)}")
    print("=" * 80)

    all_results = []
    for index, image_path in enumerate(image_paths, start=1):
        print(f"\n[{index}/{len(image_paths)}] Processing: {image_path.name}")
        try:
            context = run_pipeline(image_path)
            all_results.append(context)
            print(f"Result: {context['status']['preprocessing']}")
        except Exception as e:
            print(f"[ERROR] {image_path.name}: {e}")
            context = create_context(image_path)
            context["status"]["preprocessing"] = "failed_exception"
            all_results.append(context)

    summary_rows = []
    for context in all_results:
        image_info = context["image"]
        summary_rows.append({
            "image_name": Path(image_info["raw_path"]).name,
            "status": context["status"]["preprocessing"],
            "blur_score": image_info["blur_score"],
            "brightness_score": image_info["brightness_score"],
            "quality_score": image_info["quality_score"],
            "leaf_detected": image_info["leaf_detected"],
            "processed_path": image_info["processed_path"],
        })

    summary_df = pd.DataFrame(summary_rows)
    return all_results, summary_df

# ====================================================================
# 6. Execution
# ====================================================================
if __name__ == "__main__":
    test_dir = Path(r"Z:\Projects\Smart-Farming\Datasets\testing_images")
    all_results, summary_df = process_all_images(test_dir)

    print("\n" + "=" * 80)
    print("FINAL TEST SUMMARY")
    print("=" * 80)

    if not summary_df.empty:
        print(summary_df.to_string(index=False))

        total = len(summary_df)
        completed = (summary_df["status"] == "completed").sum()
        failed = total - completed
        leaf_detected = (summary_df["leaf_detected"] == True).sum()

        print("\n" + "=" * 80)
        print("STATISTICS")
        print("=" * 80)
        print(f"Total images         : {total}")
        print(f"Successfully processed : {completed}")
        print(f"Failed               : {failed}")
        print(f"Leaf detected        : {leaf_detected}")
        print(f"Leaf not detected    : {total - leaf_detected}")
        if total > 0:
            print(f"Success rate       : {(completed / total) * 100:.2f}%")

        print("\n" + "=" * 80)
        print("FAILURE BREAKDOWN")
        print("=" * 80)
        failure_counts = summary_df[summary_df["status"] != "completed"]["status"].value_counts()
        if failure_counts.empty:
            print("No preprocessing failures.")
        else:
            print(failure_counts.to_string())

        report_path = "preprocessing_test_results.csv"
        summary_df.to_csv(report_path, index=False)
        print(f"\nTest report saved to: {report_path}")
    else:
        print("No test results generated.")

SMART FARMING - BATCH IMAGE TEST
Testing directory : Z:\Projects\Smart-Farming\Datasets\testing_images
Total images      : 12

[1/12] Processing: aphids_tomato.jpeg
Result: completed

[2/12] Processing: army_worm_cotton.jpeg
Result: completed

[3/12] Processing: blury_leaf_1.jpg
[Validation Warning] Image too blurry (Score: 34.66 < 100.0)
[Pipeline Halted] Preprocessing failed with status: failed_blur
Result: failed_blur

[4/12] Processing: download.jpeg
Result: completed

[5/12] Processing: leaf_miner_tomato.jpeg
Result: completed

[6/12] Processing: low_brightness_leaf_1.jpg
[Validation Warning] Poor lighting (Brightness: 22.58)
[Pipeline Halted] Preprocessing failed with status: failed_lighting
Result: failed_lighting

[7/12] Processing: opencv_test_1.jpg
Result: completed

[8/12] Processing: opencv_test_2.jpg
Result: completed

[9/12] Processing: opencv_test_3.jpg
Result: completed

[10/12] Processing: opencv_test_4.jpg
Result: completed

[11/12] Processing: sample.jpg
Result: comp